In [0]:
--Data Quality Check

with tests as (

  --Check for duplicate rows
  select 'duplicate_check' as test_name,
    count(*) as failures
  from (
    select ticker, trade_date
    from analytics.mart_stock_features
    group by ticker, trade_date
    having count(*) > 1
  )

  union all

  --No Nulls in critical fields
  select 'null_check',
    count(*)
  from analytics.mart_stock_features
  where 
    ticker is null
    or trade_date is null
    or close is null

  union all

  --Valid Value Ranges
  select 'valid_values',
    count(*)
  from analytics.mart_stock_features
  where
    return_20d < -1
    or return_20d > 5
    or return_50d < -1
    or return_50d > 10

  union all

  --Volume Sanity Check
  select 'volume_check',
    count(*)
  from analytics.mart_stock_features
  where volume <= 0

  union all

  --Data is up to date
  select 'data_is_up_to_date',
    case 
      when max(trade_date) >= date_sub(current_date, 3) then 0
      else 1
    end
  from analytics.mart_stock_features

)

select *,
  case 
    when failures = 0 then 'PASS'
    else 'FAIL'
  end as status
from tests;
